# bagpus quickstart: mock recovery in minutes

**bagpus** (Bayesian Analysis of Galaxy PopUlations with Spectra) infers the *population-level* distributions of galaxy physical parameters — star formation histories, dust, metallicity — by fitting the observed distribution of an entire survey at once, using simulation-based inference (SBI) with [bagpipes](https://github.com/ACCarnall/bagpipes) as the forward model of individual galaxies.

This notebook demonstrates the full workflow on a **mock dataset**, so it needs no survey catalogue and runs in a few minutes on a laptop:

1. define a population model,
2. simulate training data,
3. train the neural posterior,
4. generate a mock observation with known parameters,
5. recover those parameters.

The simulation sizes here are deliberately tiny — results will be noisy. For science, scale up `nsims` and `ngal_sims` (see the UDS walkthrough notebook).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import bagpus

# use the stellar grids shipped with your bagpipes installation, or switch
# with bagpus.grids.change_grid(...) — see the documentation

## 1. A synthetic observed sample

`Observations` normally holds your survey: super-colours, their errors, and redshifts. Here we invent a plausible sample: redshifts uniform in $1.7<z<2$, and super-colour errors drawn to mimic a magnitude-limited survey (larger errors for redder SC1, as fainter red galaxies are noisier).

The eigenbasis files shipped in the repository (`UDS/VWSC_eigenbasis_0p5z3_wavemin2500.fits`, `FILTERS/`) define how model spectra are projected onto super-colours.

In [ ]:
rng = np.random.default_rng(42)
n_mock = 2000

redshifts = rng.uniform(1.7, 2.0, n_mock)

# fake super-colours only inform the noise model, not the fit itself
sc = np.stack([rng.uniform(-40, 90, n_mock), rng.uniform(-20, 30, n_mock)], axis=1)
sc_err = np.stack([0.5 + 0.01 * (sc[:, 0] + 50) + rng.normal(0, 0.1, n_mock),
                   0.5 + 0.005 * (sc[:, 0] + 50) + rng.normal(0, 0.05, n_mock)], axis=1)
sc_err = np.abs(sc_err) + 0.05

obs = bagpus.Observations(
    sc=sc, sc_err=sc_err, redshifts=redshifts,
    eigenbasis_file='../UDS/VWSC_eigenbasis_0p5z3_wavemin2500.fits',
    filter_list_file='../FILTERS/vwsc_uds.lis',
    filter_dir='../FILTERS/',
    n_eigenvectors=3,
    filter_mask=[9, 10],   # HST bands in the filter list not used in the data
    zmin=1.7, zmax=2.0,
)
print(len(obs), 'galaxies')

## 2. The population model

Each galaxy-level parameter is drawn from a truncated Gaussian; the *population mean and SD* of each Gaussian are the quantities we infer, with flat hyperpriors given by `mu` and `sigma`.

The `"type"` entries select the SFH and dust models from the registries in `bagpus.models` — currently `dblplaw` (double power-law with the falling slope expressed as a quenching half-life $\tau_{1/2}$) and `Calzetti` (with an Av–sSFR relation). Adding new models is a small amount of code: see the *Adding a new model* documentation page.

In [ ]:
t_zmax = obs_age = 3.5  # ~ age of Universe at z=2 (Gyr); sets the SFH peak-time prior

pop_instructions = {
    "sfh": {
        "type": "dblplaw",
        "tau":        {"limits": (0.2, 1.5 * t_zmax), "mu": (2.0, 4.0), "sigma": (0.5, 2.0)},
        "logbeta":    {"limits": (-1.0, 2.0), "mu": (-1.0, 3.0), "sigma": (0.5, 2.0)},
        "logtauhalf": {"limits": (-2.0, 1.0), "mu": (-2.0, 1.0), "sigma": (0.1, 1.0)},
    },
    "dust": {
        "type": "Calzetti",
        "eta":      {"limits": (1.0, 3.0), "mu": (1.0, 2.0), "sigma": (0.2, 2.0)},
        "logAvint": {"limits": (0.8, 2.0), "mu": (1.2, 1.5), "sigma": (0.1, 0.2)},
    },
    "metallicity": {"limits": (0.5, 2.5), "mu": (0.5, 2.0), "sigma": (0.2, 2.0)},
}

model = bagpus.PopulationModel(pop_instructions, obs)
print('population parameters:', model.param_names)
print('inferred hyperparameters:', model.n_hyper)

## 3. Simulate training data and train the posterior

`Fit` manages the workflow and caches every product under `runs/<run>/`, so re-running a cell is cheap and an interrupted session can resume.

With 60 simulations of 200 galaxies this takes a few minutes; posterior quality scales with both numbers.

In [ ]:
fit = bagpus.Fit(model, run='quickstart', ngal_sims=200, ncores=8)

fit.simulate(nsims=60)      # slow cell: ~minutes
fit.simulate_test(nsims=5)
posterior = fit.train(n_pca_components=10)

## 4. A mock observation with known parameters

We pick a "true" hyperparameter vector, forward-simulate its SC distribution, and treat that as the observed data.

In [ ]:
theta_true = np.array([
    2.5, 1.5,    # tau: mean, sd
    1.5, 1.0,    # logbeta
    0.3, 0.5,    # logtauhalf
    1.5, 0.5,    # eta
    1.4, 0.14,   # logAvint
    1.0, 0.5,    # metallicity
])

x_mock = bagpus.simulator.simulator_SC(theta_true, model, ngal=200)

plt.imshow(np.array(x_mock).T, cmap='GnBu', origin='lower', aspect='auto',
           extent=[obs.pdf_range[0][0], obs.pdf_range[0][1],
                   obs.pdf_range[1][0], obs.pdf_range[1][1]])
plt.xlabel('SC1'); plt.ylabel('SC2'); plt.title('Mock observed SC distribution');

## 5. Recover the parameters

Sample the trained posterior given the mock data, and compare with the truth (orange lines). With this tiny training set expect broad, noisy posteriors — but they should bracket the truth.

In [ ]:
samples = fit.sample_mock(x_mock, n=500)

fig = fit.plot_corner(samples=samples)
axes = np.array(fig.axes).reshape(len(theta_true), len(theta_true))
for i, truth in enumerate(theta_true):
    axes[i, i].axvline(truth, color='orange', lw=2)

## Next steps

* Fit real data: see the **UDS walkthrough** notebook, which reproduces the release-paper analysis.
* Scale up: `nsims_train=5000`, `ngal_sims=7000` were used for the paper (hours on a multi-core machine — the `scripts/` folder runs the same workflow from the command line, batch by batch).
* Different SFH or dust model: subclass `bagpus.models.SFHModel` / `DustModel` and register it.